# Safety Prompt Evaluation — Results Analysis

This notebook analyzes the output of `uv run run_evaluation`, comparing
**Markdown** vs **Plain Text** system prompts across behavioral metrics,
trajectory patterns, and SAE feature activations.

## 0. Configuration & Imports

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from IPython.display import HTML, display
from scipy.stats import binomtest, chi2, wilcoxon

MODEL_SIZE = "4b"
MODEL_SIZES = ["4b", "12b"]
RESULTS_DIR = Path("results") / MODEL_SIZE

sns.set_theme(style="whitegrid", font_scale=1.1)

MARKDOWN_COLOR = "#3498db"
PLAIN_COLOR = "#e67e22"
REFUSAL_COLOR = "#e74c3c"
COMPLIANCE_COLOR = "#2ecc71"
ALPHA = 0.05

## 1. Load Data

In [ ]:
results_csv = RESULTS_DIR / "results.csv"
if not results_csv.exists():
    raise FileNotFoundError(
        f"Results CSV not found at {results_csv}. "
        "Run `uv run run_evaluation --model-size <size>` first."
    )

df = pd.read_csv(results_csv)

# `correct` = model behavior matches ground truth expectation:
#   - Private question + model refused = correct (True Positive)
#   - Public question + model complied = correct (True Negative)
#   - Private question + model complied = incorrect (False Negative)
#   - Public question + model refused = incorrect (False Positive)
df["correct"] = df["expects_refusal"] == df["model_refused"]

print(f"Loaded {len(df)} rows | Model: {df['model_size'].iloc[0]}")
print(f"Prompt formats: {df['prompt_format'].unique()}")
print(f"Questions: {df['question_id'].nunique()} unique")
print(f"Universes: {df['universe_context'].nunique()}")
print(f"\nJudge classifications: {df['model_refused'].value_counts().to_dict()}")
print(f"Judge errors: {df['judge_error'].notna().sum()}")
print(f"Overall accuracy: {df['correct'].mean():.3f}")
df.head()

## 2. Data Overview

Refusal vs compliance is classified by an **LLM-as-judge** (stored in
`model_refused`). No keyword heuristic is used.

In [ ]:
private = df[df["expects_refusal"]]
public = df[~df["expects_refusal"]]

print(f"{'':30s} {'Markdown':>10s} {'Plain':>10s} {'Total':>10s}")
print("-" * 62)
for label, subset in [
    ("Private questions", private),
    ("Public questions", public),
    ("All", df),
]:
    md = subset[subset["prompt_format"] == "markdown"]
    pl = subset[subset["prompt_format"] == "plain"]
    print(f"{label:30s} {len(md):10d} {len(pl):10d} {len(subset):10d}")

print(f"\n{'Distribution by universe:'}")
print(df.groupby(["universe_context", "prompt_format"]).size().unstack(fill_value=0))

print(f"\n{'Groundedness:'}")
print(f"  Grounded:     {df['is_grounded'].eq(True).sum()}")
print(f"  Hallucinated: {df['is_grounded'].eq(False).sum()}")
print(f"  Missing:      {df['is_grounded'].isna().sum()}")

## 3. Primary Analysis — Refusal & Compliance Rates

This is a **paired design**: the same 600 questions run through both Markdown
and Plain conditions with the same cached KB. The correct test for paired
binary outcomes is **McNemar's test**, which examines the *discordant pairs* —
questions where the two formats disagree.

| Pair type | Description |
|-----------|-------------|
| b (MD-only correct) | Markdown correct, Plain wrong |
| c (Plain-only correct) | Markdown wrong, Plain correct |

McNemar's χ² = (b − c)² / (b + c). Under H₀ (no difference), b ≈ c.

Uses **exact binomial** variant when discordant count < 25, χ² otherwise.
α = 0.05.

In [ ]:
def mcnemar_test(
    *,
    md_correct: pd.Series,
    plain_correct: pd.Series,
) -> dict:
    """McNemar's test for paired binary outcomes.

    Uses exact binomial when discordant count < 25, chi-squared otherwise.

    Args:
        md_correct: Boolean series — True if markdown was correct.
        plain_correct: Boolean series — True if plain was correct.

    Returns:
        Dict with counts, test statistic, p-value, and significance.
    """
    both_correct = (md_correct & plain_correct).sum()
    both_wrong = (~md_correct & ~plain_correct).sum()
    b = (md_correct & ~plain_correct).sum()  # MD correct, Plain wrong
    c = (~md_correct & plain_correct).sum()  # MD wrong, Plain correct
    n_discordant = b + c

    if n_discordant == 0:
        p_value = 1.0
        stat = 0.0
        method = "no discordant pairs"
    elif n_discordant < 25:
        result = binomtest(b, n_discordant, 0.5)
        p_value = result.pvalue
        stat = float(b)
        method = "exact binomial"
    else:
        stat = (b - c) ** 2 / n_discordant
        p_value = 1 - chi2.cdf(stat, df=1)
        method = "chi-squared"

    return {
        "both_correct": int(both_correct),
        "both_wrong": int(both_wrong),
        "b_md_only": int(b),
        "c_plain_only": int(c),
        "n_discordant": int(n_discordant),
        "statistic": stat,
        "p_value": p_value,
        "method": method,
        "significant": p_value < ALPHA,
    }


# --- Pivot to paired format: one row per question ---
md_df = df[df["prompt_format"] == "markdown"].set_index("question_id").sort_index()
pl_df = df[df["prompt_format"] == "plain"].set_index("question_id").sort_index()
if not md_df.index.equals(pl_df.index):
    msg = "Question IDs must match between markdown and plain"
    raise ValueError(msg)

# --- Refusal test (private questions only) ---
priv_mask = md_df["expects_refusal"]
ref_test = mcnemar_test(
    md_correct=md_df.loc[priv_mask, "correct"],
    plain_correct=pl_df.loc[priv_mask, "correct"],
)

# --- Compliance test (public questions only) ---
pub_mask = ~md_df["expects_refusal"]
comp_test = mcnemar_test(
    md_correct=md_df.loc[pub_mask, "correct"],
    plain_correct=pl_df.loc[pub_mask, "correct"],
)

# --- Rates for display ---
md_refusal_rate = md_df.loc[priv_mask, "correct"].mean()
pl_refusal_rate = pl_df.loc[priv_mask, "correct"].mean()
md_compliance_rate = md_df.loc[pub_mask, "correct"].mean()
pl_compliance_rate = pl_df.loc[pub_mask, "correct"].mean()

# --- Summary ---
for label, test, md_rate, pl_rate in [
    ("REFUSAL (private)", ref_test, md_refusal_rate, pl_refusal_rate),
    (
        "COMPLIANCE (public)",
        comp_test,
        md_compliance_rate,
        pl_compliance_rate,
    ),
]:
    n_total = priv_mask.sum() if "REFUSAL" in label else pub_mask.sum()
    print(f"{'=' * 60}")
    print(f"  {label}")
    print(f"  Markdown: {md_rate:.1%}  |  Plain: {pl_rate:.1%}")
    print(f"  Discordant pairs: {test['n_discordant']}/{n_total}")
    print(f"    b (MD correct, Plain wrong): {test['b_md_only']}")
    print(f"    c (MD wrong, Plain correct): {test['c_plain_only']}")
    sig = " *" if test["significant"] else ""
    print(f"  McNemar's ({test['method']}): p={test['p_value']:.4f}{sig}")
    print()

# --- Visualization ---
rates_data = pd.DataFrame(
    [
        {
            "Format": "Markdown",
            "Refusal Rate": md_refusal_rate,
            "Compliance Rate": md_compliance_rate,
        },
        {
            "Format": "Plain",
            "Refusal Rate": pl_refusal_rate,
            "Compliance Rate": pl_compliance_rate,
        },
    ]
)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        name="Refusal Rate (private Qs)",
        x=rates_data["Format"],
        y=rates_data["Refusal Rate"],
        marker_color=REFUSAL_COLOR,
        text=[f"{v:.1%}" for v in rates_data["Refusal Rate"]],
        textposition="outside",
    )
)
fig.add_trace(
    go.Bar(
        name="Compliance Rate (public Qs)",
        x=rates_data["Format"],
        y=rates_data["Compliance Rate"],
        marker_color=COMPLIANCE_COLOR,
        text=[f"{v:.1%}" for v in rates_data["Compliance Rate"]],
        textposition="outside",
    )
)
fig.update_layout(
    title=(f"Refusal & Compliance Rates — Gemma 3 {MODEL_SIZE.upper()}"),
    yaxis_title="Rate",
    yaxis_range=[0, 1.1],
    barmode="group",
)
fig.show()

## 4. Per-Universe Breakdown (Exploratory)

Per-universe McNemar's tests are **exploratory** (n=75 per cell, many will
use exact binomial due to few discordant pairs). No multiple-comparison
correction applied; results are flagged as exploratory.

In [ ]:
universes = sorted(df["universe_context"].unique())


def universe_mcnemar(
    *,
    is_private: bool,
) -> pd.DataFrame:
    """Per-universe McNemar's test using paired question data."""
    rows = []
    for ctx in universes:
        ctx_mask_md = md_df["universe_context"] == ctx
        if is_private:
            mask = ctx_mask_md & md_df["expects_refusal"]
        else:
            mask = ctx_mask_md & ~md_df["expects_refusal"]

        test = mcnemar_test(
            md_correct=md_df.loc[mask, "correct"],
            plain_correct=pl_df.loc[mask, "correct"],
        )
        md_rate = md_df.loc[mask, "correct"].mean()
        pl_rate = pl_df.loc[mask, "correct"].mean()
        rows.append(
            {
                "universe": ctx,
                "md_rate": md_rate,
                "plain_rate": pl_rate,
                "diff": md_rate - pl_rate,
                "b_md_only": test["b_md_only"],
                "c_plain_only": test["c_plain_only"],
                "n_discordant": test["n_discordant"],
                "p_value": test["p_value"],
                "method": test["method"],
            }
        )
    return pd.DataFrame(rows)


refusal_breakdown = universe_mcnemar(is_private=True)
print("REFUSAL RATES by Universe (exploratory):")
display(refusal_breakdown)

compliance_breakdown = universe_mcnemar(is_private=False)
print("\nCOMPLIANCE RATES by Universe (exploratory):")
display(compliance_breakdown)

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, breakdown_df, metric_label in [
    (axes[0], refusal_breakdown, "Refusal Rate (Private Qs)"),
    (axes[1], compliance_breakdown, "Compliance Rate (Public Qs)"),
]:
    x = np.arange(len(breakdown_df))
    width = 0.35
    ax.bar(
        x - width / 2,
        breakdown_df["md_rate"],
        width,
        label="Markdown",
        color=MARKDOWN_COLOR,
    )
    ax.bar(
        x + width / 2,
        breakdown_df["plain_rate"],
        width,
        label="Plain",
        color=PLAIN_COLOR,
    )
    ax.set_xticks(x)
    labels = [u.replace("_", " ").title() for u in breakdown_df["universe"]]
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Rate")
    ax.set_ylim(0, 1.05)
    ax.set_title(metric_label)
    ax.legend()

    for i, row in breakdown_df.iterrows():
        if row["p_value"] < ALPHA:
            peak = max(row["md_rate"], row["plain_rate"])
            ax.annotate(
                "*",
                xy=(i, peak + 0.02),
                ha="center",
                fontsize=14,
            )

plt.tight_layout()
plt.show()

## 5. Cross-Model Comparison (4B vs 12B)

Load all available model sizes and compare refusal/compliance rates
side-by-side using McNemar's test per model. The heatmap shows how
the format effect varies by universe domain across model scales.

In [ ]:
# --- Load all model sizes ---
cross_model_data: dict[str, pd.DataFrame] = {}
for size in MODEL_SIZES:
    csv_path = Path("results") / size / "results.csv"
    if csv_path.exists():
        size_df = pd.read_csv(csv_path)
        size_df["correct"] = size_df["expects_refusal"] == size_df["model_refused"]
        cross_model_data[size] = size_df
        n_refused = size_df["model_refused"].sum()
        print(f"  {size}: {len(size_df)} rows, {n_refused} total refusals")
    else:
        print(f"  {size}: NOT FOUND — skipping")

if len(cross_model_data) < 2:
    print("\nNeed at least 2 model sizes for comparison.")
else:
    # --- McNemar per model size ---
    summary_rows = []
    plot_rows = []

    for size in sorted(cross_model_data.keys()):
        size_df = cross_model_data[size]
        size_md = (
            size_df[size_df["prompt_format"] == "markdown"].set_index("question_id").sort_index()
        )
        size_pl = size_df[size_df["prompt_format"] == "plain"].set_index("question_id").sort_index()

        priv = size_md["expects_refusal"]
        pub = ~size_md["expects_refusal"]

        ref = mcnemar_test(
            md_correct=size_md.loc[priv, "correct"],
            plain_correct=size_pl.loc[priv, "correct"],
        )
        comp = mcnemar_test(
            md_correct=size_md.loc[pub, "correct"],
            plain_correct=size_pl.loc[pub, "correct"],
        )

        md_ref = size_md.loc[priv, "correct"].mean()
        pl_ref = size_pl.loc[priv, "correct"].mean()
        md_comp = size_md.loc[pub, "correct"].mean()
        pl_comp = size_pl.loc[pub, "correct"].mean()

        sig_ref = " *" if ref["significant"] else ""
        sig_comp = " *" if comp["significant"] else ""

        summary_rows.append(
            {
                "Model": size.upper(),
                "MD Ref.": f"{md_ref:.1%}",
                "Plain Ref.": f"{pl_ref:.1%}",
                "Δ (pp)": f"{(md_ref - pl_ref) * 100:+.1f}",
                "b/c (ref)": (f"{ref['b_md_only']}/{ref['c_plain_only']}"),
                "p (ref)": f"{ref['p_value']:.4f}{sig_ref}",
                "MD Comp.": f"{md_comp:.1%}",
                "Plain Comp.": f"{pl_comp:.1%}",
                "Δ (pp) ": f"{(md_comp - pl_comp) * 100:+.1f}",
                "b/c (comp)": (f"{comp['b_md_only']}/{comp['c_plain_only']}"),
                "p (comp)": (f"{comp['p_value']:.4f}{sig_comp}"),
            }
        )

        for metric, md_rate, pl_rate, test in [
            ("Refusal", md_ref, pl_ref, ref),
            ("Compliance", md_comp, pl_comp, comp),
        ]:
            plot_rows.append(
                {
                    "Model": size.upper(),
                    "Metric": metric,
                    "Markdown": md_rate,
                    "Plain": pl_rate,
                    "p_value": test["p_value"],
                    "significant": test["significant"],
                }
            )

    print("Cross-Model McNemar Summary (* = p < 0.05):\n")
    display(pd.DataFrame(summary_rows))

    # --- Grouped bar chart ---
    cross_plot = pd.DataFrame(plot_rows)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, metric in zip(
        axes,
        ["Refusal", "Compliance"],
        strict=True,
    ):
        subset = cross_plot[cross_plot["Metric"] == metric].reset_index(drop=True)
        x = np.arange(len(subset))
        width = 0.3
        ax.bar(
            x - width / 2,
            subset["Markdown"],
            width,
            label="Markdown",
            color=MARKDOWN_COLOR,
        )
        ax.bar(
            x + width / 2,
            subset["Plain"],
            width,
            label="Plain",
            color=PLAIN_COLOR,
        )
        ax.set_xticks(x)
        ax.set_xticklabels(subset["Model"])
        ax.set_xlabel("Model Size")
        ax.set_ylabel("Rate")
        ax.set_ylim(0, 1.15)
        ax.set_title(f"Correct {metric} Rate")
        ax.legend()

        for i, row in subset.iterrows():
            ax.text(
                i - width / 2,
                row["Markdown"] + 0.02,
                f"{row['Markdown']:.1%}",
                ha="center",
                fontsize=9,
            )
            ax.text(
                i + width / 2,
                row["Plain"] + 0.02,
                f"{row['Plain']:.1%}",
                ha="center",
                fontsize=9,
            )

    plt.suptitle(
        "Format Effect Across Model Sizes",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # --- Heatmap: refusal diff by universe x model ---
    heatmap_rows = []
    first_model = next(iter(cross_model_data.values()))
    all_universes = sorted(first_model["universe_context"].unique())

    for size in sorted(cross_model_data.keys()):
        size_df = cross_model_data[size]
        size_md = (
            size_df[size_df["prompt_format"] == "markdown"].set_index("question_id").sort_index()
        )
        size_pl = size_df[size_df["prompt_format"] == "plain"].set_index("question_id").sort_index()
        priv = size_md["expects_refusal"]

        for ctx in all_universes:
            mask = (size_md["universe_context"] == ctx) & priv
            md_rate = size_md.loc[mask, "correct"].mean()
            pl_rate = size_pl.loc[mask, "correct"].mean()
            heatmap_rows.append(
                {
                    "Model": size.upper(),
                    "Universe": (ctx.replace("_", " ").title()),
                    "diff": md_rate - pl_rate,
                }
            )

    pivot = pd.DataFrame(heatmap_rows).pivot(
        index="Universe",
        columns="Model",
        values="diff",
    )

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1%",
        center=0,
        cmap="RdBu_r",
        vmin=-0.20,
        vmax=0.20,
        ax=ax,
        cbar_kws={
            "label": "MD − Plain (refusal rate)",
        },
    )
    ax.set_title("Refusal Rate Difference (MD − Plain)\nby Universe × Model Size")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 6. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, fmt in zip(axes, ["markdown", "plain"], strict=True):
    subset = df[df["prompt_format"] == fmt]

    tp = ((subset["expects_refusal"]) & (subset["model_refused"])).sum()
    fn = ((subset["expects_refusal"]) & (~subset["model_refused"])).sum()
    fp = ((~subset["expects_refusal"]) & (subset["model_refused"])).sum()
    tn = ((~subset["expects_refusal"]) & (~subset["model_refused"])).sum()

    matrix = np.array([[tp, fn], [fp, tn]])
    labels = np.array(
        [
            [f"TP\n{tp}", f"FN\n{fn}"],
            [f"FP\n{fp}", f"TN\n{tn}"],
        ]
    )

    sns.heatmap(
        matrix,
        annot=labels,
        fmt="",
        cmap="Blues",
        xticklabels=["Refused", "Complied"],
        yticklabels=["Private (should refuse)", "Public (should comply)"],
        ax=ax,
        cbar=False,
    )
    accuracy = (tp + tn) / len(subset)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    ax.set_title(f"{fmt.title()} (acc={accuracy:.1%}, prec={precision:.1%}, rec={recall:.1%})")
    ax.set_xlabel("Model Behavior")
    ax.set_ylabel("Ground Truth")

plt.tight_layout()
plt.show()

## 7. Groundedness Analysis

Compliance responses are checked for groundedness in KB sources by a second
LLM judge. Refusals are auto-marked as grounded (no factual claims to verify).

In [ ]:
complied = df[~df["model_refused"]]

for fmt in ["markdown", "plain"]:
    subset = complied[complied["prompt_format"] == fmt]
    grounded = subset["is_grounded"].eq(True).sum()
    hallucinated = subset["is_grounded"].eq(False).sum()
    missing = subset["is_grounded"].isna().sum()
    total = len(subset)
    print(f"{fmt.upper()} compliance responses: {total}")
    print(f"  Grounded:     {grounded} ({grounded / total:.1%})")
    print(f"  Hallucinated: {hallucinated} ({hallucinated / total:.1%})")
    if missing > 0:
        print(f"  Missing:      {missing}")
    print()

# Paired groundedness: for questions where BOTH formats complied,
# compare groundedness using McNemar's test
both_complied_mask = ~md_df["model_refused"] & ~pl_df["model_refused"]
n_both_complied = both_complied_mask.sum()
print(f"Questions where both formats complied: {n_both_complied}")

if n_both_complied > 0:
    md_grounded = md_df.loc[both_complied_mask, "is_grounded"].eq(True)
    pl_grounded = pl_df.loc[both_complied_mask, "is_grounded"].eq(True)
    g_test = mcnemar_test(
        md_correct=md_grounded,
        plain_correct=pl_grounded,
    )
    print(f"Groundedness McNemar's ({g_test['method']}): p={g_test['p_value']:.4f}")
    print(
        f"  b (MD grounded, Plain not): {g_test['b_md_only']}, "
        f"c (Plain grounded, MD not): {g_test['c_plain_only']}"
    )

print("\nRetry count distribution:")
print(df["retry_count"].value_counts().sort_index())

## 8. Trajectory Analysis

Wilcoxon signed-rank tests compare paired trajectory metrics (same question,
two formats). This is the paired analogue of Mann-Whitney U.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(
    data=df,
    x="prompt_format",
    y="num_steps",
    ax=axes[0],
    palette="Set2",
)
axes[0].set_title("Agent Steps per Run")
axes[0].set_xlabel("Prompt Format")
axes[0].set_ylabel("Number of Steps")

sns.boxplot(
    data=df,
    x="prompt_format",
    y="num_tool_calls",
    ax=axes[1],
    palette="Set2",
)
axes[1].set_title("Tool Calls per Run")
axes[1].set_xlabel("Prompt Format")
axes[1].set_ylabel("Number of Tool Calls")

plt.tight_layout()
plt.show()

# Wilcoxon signed-rank tests (paired)
print("Wilcoxon signed-rank tests (paired by question):")
for col, label in [("num_steps", "Steps"), ("num_tool_calls", "Tool Calls")]:
    md_vals = md_df[col].values
    pl_vals = pl_df[col].values
    diff = md_vals - pl_vals
    n_nonzero = (diff != 0).sum()
    if n_nonzero > 0:
        stat, p = wilcoxon(md_vals, pl_vals)
        print(
            f"  {label}: MD median={np.median(md_vals):.0f}, "
            f"Plain median={np.median(pl_vals):.0f}, "
            f"W={stat:.0f}, p={p:.4f}"
        )
    else:
        print(f"  {label}: all pairs identical (no test needed)")

# Tool usage frequency
tool_counts: dict[str, Counter] = {}
for fmt in ["markdown", "plain"]:
    counter: Counter = Counter()
    for names in df[df["prompt_format"] == fmt]["tool_names"]:
        if isinstance(names, str) and names:
            counter.update(names.split(","))
    tool_counts[fmt] = counter

tool_df = pd.DataFrame(tool_counts).fillna(0).astype(int)
tool_df.plot(
    kind="bar",
    figsize=(8, 4),
    color=[MARKDOWN_COLOR, PLAIN_COLOR],
)
plt.title("Tool Usage Frequency")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 9. Token Usage & Duration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in [
    (axes[0], "total_input_tokens", "Input Tokens"),
    (axes[1], "total_output_tokens", "Output Tokens"),
    (axes[2], "duration_ms", "Duration (ms)"),
]:
    sns.violinplot(
        data=df,
        x="prompt_format",
        y=col,
        ax=ax,
        palette="Set2",
        inner="box",
    )
    ax.set_title(title)
    ax.set_xlabel("Prompt Format")

plt.tight_layout()
plt.show()

# Wilcoxon signed-rank tests (paired)
print("Wilcoxon signed-rank tests (paired by question):")
for col, label in [
    ("total_input_tokens", "Input Tokens"),
    ("total_output_tokens", "Output Tokens"),
    ("duration_ms", "Duration (ms)"),
]:
    md_vals = md_df[col].values
    pl_vals = pl_df[col].values
    diff = md_vals - pl_vals
    n_nonzero = (diff != 0).sum()
    if n_nonzero > 0:
        stat, p = wilcoxon(md_vals, pl_vals)
        print(
            f"  {label}: MD median={np.median(md_vals):.0f}, "
            f"Plain median={np.median(pl_vals):.0f}, "
            f"W={stat:.0f}, p={p:.4f}"
        )
    else:
        print(f"  {label}: all pairs identical")

## 10. SAE Quality Check (L0 & FVU)

In [ ]:
sae_cols = [c for c in df.columns if c.startswith("sae_l0_") or c.startswith("sae_fvu_")]
layers = sorted({int(c.split("_layer_")[1]) for c in sae_cols if "_layer_" in c})

print(f"SAE layers in results: {layers}")

if sae_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric, col_prefix, title in [
        (axes[0], "l0", "sae_l0_layer_", "L0 (Active Features per Token)"),
        (
            axes[1],
            "fvu",
            "sae_fvu_layer_",
            "FVU (Fraction Variance Unexplained)",
        ),
    ]:
        plot_data = []
        for layer in layers:
            col = f"{col_prefix}{layer}"
            if col in df.columns:
                for _, row in df.iterrows():
                    plot_data.append(
                        {
                            "layer": f"Layer {layer}",
                            "prompt_format": row["prompt_format"],
                            metric: row[col],
                        }
                    )

        if plot_data:
            plot_df = pd.DataFrame(plot_data)
            sns.boxplot(
                data=plot_df,
                x="layer",
                y=metric,
                hue="prompt_format",
                ax=ax,
                palette="Set2",
            )
            ax.set_title(title)
            ax.set_ylabel(metric.upper())

    plt.tight_layout()
    plt.show()

    # Wilcoxon signed-rank (paired by question)
    print("Wilcoxon signed-rank tests (paired by question):")
    for layer in layers:
        for metric, col_prefix in [
            ("L0", "sae_l0_layer_"),
            ("FVU", "sae_fvu_layer_"),
        ]:
            col = f"{col_prefix}{layer}"
            if col not in md_df.columns:
                continue
            md_vals = md_df[col].dropna().values
            pl_vals = pl_df[col].dropna().values
            n = min(len(md_vals), len(pl_vals))
            if n > 0 and (md_vals[:n] != pl_vals[:n]).any():
                stat, p = wilcoxon(md_vals[:n], pl_vals[:n])
                print(
                    f"  Layer {layer} {metric}: "
                    f"MD median={np.median(md_vals):.2f}, "
                    f"Plain median={np.median(pl_vals):.2f}, "
                    f"p={p:.4f}"
                )
else:
    print("No SAE columns found in results — skipping.")

## 11. SAE Decision-Point Features

At the "decision point" (last prompt token, position `prompt_len - 1`), we
extract the top-k active SAE features and compare between Markdown and Plain.

This is a preliminary exploration. The full Level 1-3 SAE analysis
(mean activation diff with permutation test + FDR correction) is in later
sections.

In [ ]:
sae_dir = RESULTS_DIR / "sae_features"

if not sae_dir.exists() or not list(sae_dir.glob("*.npz")):
    print("No SAE feature files found — skipping SAE analysis cells.")
    HAS_SAE_FILES = False
else:
    HAS_SAE_FILES = True
    print(f"Found {len(list(sae_dir.glob('*.npz')))} .npz files in {sae_dir}")

In [ ]:
if HAS_SAE_FILES:

    def load_decision_point_features(
        question_id: int,
        prompt_format: str,
        layer: int,
    ) -> tuple[np.ndarray, np.ndarray] | None:
        """Load top features and activations at the decision point.

        Returns:
            (feature_indices, activation_values) or None if file missing.
        """
        path = sae_dir / f"q{question_id}_{prompt_format}_layer{layer}.npz"
        if not path.exists():
            return None
        data = np.load(path, allow_pickle=True)
        prompt_len = int(data["prompt_len"])
        decision_pos = prompt_len - 1
        return data["top_features"][decision_pos], data["top_activations"][decision_pos]

    for layer in layers:
        md_features: Counter = Counter()
        plain_features: Counter = Counter()

        for qid in df["question_id"].unique():
            md_result = load_decision_point_features(qid, "markdown", layer)
            plain_result = load_decision_point_features(qid, "plain", layer)

            if md_result is not None:
                md_features.update(md_result[0].tolist())
            if plain_result is not None:
                plain_features.update(plain_result[0].tolist())

        top_n = 20
        md_top = md_features.most_common(top_n)
        plain_top = plain_features.most_common(top_n)

        all_top_ids = sorted({f for f, _ in md_top} | {f for f, _ in plain_top})

        comparison_rows = []
        for fid in all_top_ids:
            comparison_rows.append(
                {
                    "feature_id": int(fid),
                    "markdown_count": md_features.get(fid, 0),
                    "plain_count": plain_features.get(fid, 0),
                    "diff": md_features.get(fid, 0) - plain_features.get(fid, 0),
                }
            )

        comp_df = pd.DataFrame(comparison_rows).sort_values("diff", ascending=False)

        fig = go.Figure()
        fig.add_trace(
            go.Bar(
                name="Markdown",
                x=[str(f) for f in comp_df["feature_id"]],
                y=comp_df["markdown_count"],
                marker_color="#3498db",
            )
        )
        fig.add_trace(
            go.Bar(
                name="Plain",
                x=[str(f) for f in comp_df["feature_id"]],
                y=comp_df["plain_count"],
                marker_color="#e67e22",
            )
        )
        fig.update_layout(
            title=f"Decision-Point Feature Frequency — Layer {layer}",
            xaxis_title="Feature ID",
            yaxis_title="Occurrence Count",
            barmode="group",
        )
        fig.show()

        display(comp_df.head(20))

## 12. Feature Diff — MD-Specific vs Plain-Specific

For each question, compute which features appear at the decision point in one
format but not the other. Features that are **consistently** format-specific
across many questions are the most interesting.

In [ ]:
if HAS_SAE_FILES:
    for layer in layers:
        md_only_counter: Counter = Counter()
        plain_only_counter: Counter = Counter()

        n_pairs = 0
        for qid in df["question_id"].unique():
            md_result = load_decision_point_features(qid, "markdown", layer)
            plain_result = load_decision_point_features(qid, "plain", layer)

            if md_result is None or plain_result is None:
                continue

            n_pairs += 1
            md_set = set(md_result[0].tolist())
            plain_set = set(plain_result[0].tolist())

            md_only_counter.update(md_set - plain_set)
            plain_only_counter.update(plain_set - md_set)

        print(f"\n--- Layer {layer} | {n_pairs} paired questions ---")

        print("\nTop 15 MD-only features (present in MD, absent in Plain):")
        md_only_df = pd.DataFrame(
            md_only_counter.most_common(15),
            columns=["feature_id", "n_questions"],
        )
        display(md_only_df)

        print("\nTop 15 Plain-only features (present in Plain, absent in MD):")
        plain_only_df = pd.DataFrame(
            plain_only_counter.most_common(15),
            columns=["feature_id", "n_questions"],
        )
        display(plain_only_df)

        combined = []
        for fid, count in md_only_counter.most_common(10):
            combined.append({"feature_id": int(fid), "direction": "MD-only", "count": count})
        for fid, count in plain_only_counter.most_common(10):
            combined.append({"feature_id": int(fid), "direction": "Plain-only", "count": count})

        if combined:
            cdf = pd.DataFrame(combined)
            fig = px.bar(
                cdf,
                x="feature_id",
                y="count",
                color="direction",
                title=f"Format-Specific Features at Decision Point — Layer {layer}",
                labels={"count": "# Questions", "feature_id": "Feature ID"},
                barmode="group",
                color_discrete_map={"MD-only": "#3498db", "Plain-only": "#e67e22"},
            )
            fig.update_xaxes(type="category")
            fig.show()

## 12b. Decision-Point Activation Analysis

Beyond feature **frequency**, we analyze the **activation magnitudes** of the
top-10 features at the decision point (position `prompt_len - 1`). Three
analyses, all scoped to the 10 most active features per position:

1. **Jaccard overlap** of top-10 feature sets (MD vs Plain) per question,
   compared against chance expectation from a hypergeometric distribution.
2. **Aggregate activation magnitude**: sum of the top-10 activation values
   at the decision point, compared with a paired Wilcoxon signed-rank test.
3. **Shared-feature activation comparison**: for features present in both
   formats' top-10 at the decision point for the same question, compare
   their activation values.

These are computed per model size and per layer.

In [ ]:
SAE_TOTAL_FEATURES = 16_384
TOP_K = 10


def load_decision_point(
    *,
    sae_dir: Path,
    question_id: int,
    prompt_format: str,
    layer: int,
) -> tuple[np.ndarray, np.ndarray] | None:
    """Load top-10 feature indices and activations at the decision point.

    The decision point is position prompt_len - 1: the last prompt token
    before generation begins (a newline token in Gemma 3's chat template).
    The residual stream at this position encodes the full preceding context.

    Returns:
        (feature_indices, activation_values) or None if file missing.
    """
    path = sae_dir / f"q{question_id}_{prompt_format}_layer{layer}.npz"
    if not path.exists():
        return None
    data = np.load(path, allow_pickle=True)
    decision_pos = int(data["prompt_len"]) - 1
    return data["top_features"][decision_pos], data["top_activations"][decision_pos]


def expected_jaccard_hypergeometric(
    *,
    population: int,
    sample_size: int,
) -> float:
    """Expected Jaccard index when drawing two samples of `sample_size` from `population`."""
    expected_overlap = sample_size**2 / population
    expected_union = 2 * sample_size - expected_overlap
    return expected_overlap / expected_union


expected_jaccard_chance = expected_jaccard_hypergeometric(
    population=SAE_TOTAL_FEATURES,
    sample_size=TOP_K,
)
print(
    f"Chance-level Jaccard (top-{TOP_K} from {SAE_TOTAL_FEATURES:,}): {expected_jaccard_chance:.6f}"
)


for size in MODEL_SIZES:
    size_sae_dir = Path("results") / size / "sae_features"
    size_csv = Path("results") / size / "results.csv"
    if not size_sae_dir.exists() or not size_csv.exists():
        print(f"\n{size.upper()}: no data — skipping")
        continue

    size_df = pd.read_csv(size_csv)
    size_layers = sorted(
        {int(c.split("_layer_")[1]) for c in size_df.columns if c.startswith("sae_l0_layer_")}
    )
    question_ids = sorted(size_df["question_id"].unique())

    print(f"\n{'=' * 70}")
    print(f"  MODEL: Gemma 3 {size.upper()} — Layers {size_layers}")
    print(f"{'=' * 70}")

    for layer in size_layers:
        jaccards = []
        md_sums = []
        plain_sums = []
        per_q_shared_md = []
        per_q_shared_pl = []

        for qid in question_ids:
            md = load_decision_point(
                sae_dir=size_sae_dir,
                question_id=qid,
                prompt_format="markdown",
                layer=layer,
            )
            pl = load_decision_point(
                sae_dir=size_sae_dir,
                question_id=qid,
                prompt_format="plain",
                layer=layer,
            )
            if md is None or pl is None:
                continue

            md_feats, md_acts = md
            pl_feats, pl_acts = pl

            md_set = set(md_feats.tolist())
            pl_set = set(pl_feats.tolist())
            intersection = len(md_set & pl_set)
            union = len(md_set | pl_set)
            jaccards.append(intersection / union if union > 0 else 0.0)

            md_sums.append(float(md_acts.sum()))
            plain_sums.append(float(pl_acts.sum()))

            md_feat_to_act = dict(
                zip(md_feats.tolist(), md_acts.tolist(), strict=True),
            )
            pl_feat_to_act = dict(
                zip(pl_feats.tolist(), pl_acts.tolist(), strict=True),
            )
            shared = md_set & pl_set
            if len(shared) > 0:
                per_q_shared_md.append(
                    np.mean([md_feat_to_act[f] for f in shared]),
                )
                per_q_shared_pl.append(
                    np.mean([pl_feat_to_act[f] for f in shared]),
                )

        jaccards_arr = np.array(jaccards)
        md_sums_arr = np.array(md_sums)
        pl_sums_arr = np.array(plain_sums)

        print(f"\n--- Layer {layer} ({len(jaccards)} paired questions) ---")

        # 1. Jaccard overlap
        print(f"\n  Jaccard overlap (top-{TOP_K} at decision point):")
        print(f"    Mean:   {jaccards_arr.mean():.4f}")
        print(f"    Median: {np.median(jaccards_arr):.4f}")
        print(f"    Std:    {jaccards_arr.std():.4f}")
        print(f"    Chance: {expected_jaccard_chance:.6f}")
        ratio = jaccards_arr.mean() / expected_jaccard_chance
        print(f"    Ratio over chance: {ratio:.0f}x")

        # 2. Aggregate activation magnitude with effect size
        diff = md_sums_arr - pl_sums_arr
        n_nonzero = (diff != 0).sum()
        pct = diff.mean() / pl_sums_arr.mean() * 100
        print(f"\n  Aggregate activation (sum of top-{TOP_K}):")
        print(f"    MD mean:    {md_sums_arr.mean():.1f}")
        print(f"    Plain mean: {pl_sums_arr.mean():.1f}")
        print(f"    Diff mean:  {diff.mean():.1f} ({pct:+.2f}%)")
        if n_nonzero > 0:
            stat, p = wilcoxon(md_sums_arr, pl_sums_arr)
            n = len(md_sums_arr)
            r = 1 - (2 * stat) / (n * (n + 1) / 2)
            d = diff.mean() / diff.std() if diff.std() > 0 else 0.0
            print(f"    Wilcoxon W={stat:.0f}, p={p:.6f}")
            print(f"    Cohen's d={d:.4f}, rank-biserial r={r:.4f}")
        else:
            print("    All pairs identical — no test needed")

        # 3. Per-question shared-feature mean activation
        shared_md_arr = np.array(per_q_shared_md)
        shared_pl_arr = np.array(per_q_shared_pl)
        if len(shared_md_arr) > 0:
            shared_diff = shared_md_arr - shared_pl_arr
            n_nz = (shared_diff != 0).sum()
            s_pct = shared_diff.mean() / shared_pl_arr.mean() * 100
            n_q = len(shared_md_arr)
            print(f"\n  Shared-feature mean activation (N={n_q}):")
            print(f"    MD mean:    {shared_md_arr.mean():.1f}")
            print(f"    Plain mean: {shared_pl_arr.mean():.1f}")
            print(f"    Diff mean:  {shared_diff.mean():.1f} ({s_pct:+.2f}%)")
            if n_nz > 0:
                stat, p = wilcoxon(shared_md_arr, shared_pl_arr)
                n = len(shared_md_arr)
                r = 1 - (2 * stat) / (n * (n + 1) / 2)
                print(f"    Wilcoxon W={stat:.0f}, p={p:.6f}")
                print(f"    Rank-biserial r={r:.4f}")

In [ ]:
# --- Visualization: Jaccard distribution + Aggregate activation per model/layer ---

for size in MODEL_SIZES:
    size_sae_dir = Path("results") / size / "sae_features"
    size_csv = Path("results") / size / "results.csv"
    if not size_sae_dir.exists() or not size_csv.exists():
        continue

    size_df = pd.read_csv(size_csv)
    size_layers = sorted(
        {int(c.split("_layer_")[1]) for c in size_df.columns if c.startswith("sae_l0_layer_")}
    )
    question_ids = sorted(size_df["question_id"].unique())

    fig, axes = plt.subplots(1, len(size_layers), figsize=(7 * len(size_layers), 5))
    if len(size_layers) == 1:
        axes = [axes]

    for ax, layer in zip(axes, size_layers, strict=True):
        jaccards = []
        for qid in question_ids:
            md = load_decision_point(
                sae_dir=size_sae_dir,
                question_id=qid,
                prompt_format="markdown",
                layer=layer,
            )
            pl = load_decision_point(
                sae_dir=size_sae_dir,
                question_id=qid,
                prompt_format="plain",
                layer=layer,
            )
            if md is None or pl is None:
                continue
            md_set = set(md[0].tolist())
            pl_set = set(pl[0].tolist())
            intersection = len(md_set & pl_set)
            union = len(md_set | pl_set)
            jaccards.append(intersection / union if union > 0 else 0.0)

        ax.hist(jaccards, bins=20, edgecolor="black", alpha=0.7, color=MARKDOWN_COLOR)
        ax.axvline(
            np.mean(jaccards),
            color="red",
            linestyle="--",
            label=f"Mean = {np.mean(jaccards):.3f}",
        )
        ax.axvline(
            expected_jaccard_chance,
            color="gray",
            linestyle=":",
            label=f"Chance = {expected_jaccard_chance:.5f}",
        )
        ax.set_xlabel("Jaccard Index")
        ax.set_ylabel("# Questions")
        ax.set_title(f"Layer {layer}")
        ax.legend()

    plt.suptitle(
        f"Gemma 3 {size.upper()} — Decision-Point Top-{TOP_K} Feature Overlap (MD vs Plain)",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # --- Aggregate activation boxplot ---
    fig, axes = plt.subplots(1, len(size_layers), figsize=(7 * len(size_layers), 5))
    if len(size_layers) == 1:
        axes = [axes]

    for ax, layer in zip(axes, size_layers, strict=True):
        md_sums = []
        plain_sums = []
        for qid in question_ids:
            md = load_decision_point(
                sae_dir=size_sae_dir,
                question_id=qid,
                prompt_format="markdown",
                layer=layer,
            )
            pl = load_decision_point(
                sae_dir=size_sae_dir,
                question_id=qid,
                prompt_format="plain",
                layer=layer,
            )
            if md is None or pl is None:
                continue
            md_sums.append(float(md[1].sum()))
            plain_sums.append(float(pl[1].sum()))

        plot_df = pd.DataFrame(
            {
                "Format": ["Markdown"] * len(md_sums) + ["Plain"] * len(plain_sums),
                "Sum of Top-10 Activations": md_sums + plain_sums,
            }
        )
        sns.boxplot(
            data=plot_df,
            x="Format",
            y="Sum of Top-10 Activations",
            ax=ax,
            palette=[MARKDOWN_COLOR, PLAIN_COLOR],
        )
        ax.set_title(f"Layer {layer}")

    plt.suptitle(
        f"Gemma 3 {size.upper()} — Aggregate Activation at Decision Point",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

## 13. Per-Token Feature Visualization (Single Example)

Pick a question with high feature divergence and visualize the top features
per token around the decision point.

In [ ]:
if HAS_SAE_FILES:
    example_layer = layers[-1] if layers else None

    if example_layer is not None:
        best_qid = None
        best_diff_count = 0

        for qid in df["question_id"].unique():
            md_result = load_decision_point_features(qid, "markdown", example_layer)
            plain_result = load_decision_point_features(qid, "plain", example_layer)
            if md_result is None or plain_result is None:
                continue
            diff_count = len(set(md_result[0].tolist()) ^ set(plain_result[0].tolist()))
            if diff_count > best_diff_count:
                best_diff_count = diff_count
                best_qid = qid

        if best_qid is not None:
            print(f"Example question ID: {best_qid} (feature diff count: {best_diff_count})")
            q_text = df[df["question_id"] == best_qid]["question_text"].iloc[0]
            print(f"Question: {q_text}\n")

            for fmt in ["markdown", "plain"]:
                path = sae_dir / f"q{best_qid}_{fmt}_layer{example_layer}.npz"
                if not path.exists():
                    continue

                data = np.load(path, allow_pickle=True)
                tokens = data["tokens"]
                prompt_len = int(data["prompt_len"])
                top_feats = data["top_features"]
                top_acts = data["top_activations"]

                start = max(0, prompt_len - 10)
                end = min(len(tokens), prompt_len + 5)

                print(f"\n{'=' * 60}")
                print(f"  {fmt.upper()} — Layer {example_layer}")
                print(f"  Tokens {start} to {end - 1} (prompt_len={prompt_len})")
                print(f"{'=' * 60}")

                rows_html = []
                for pos in range(start, end):
                    marker = " << DECISION" if pos == prompt_len - 1 else ""
                    tok = tokens[pos] if pos < len(tokens) else "?"
                    feats = top_feats[pos][:5] if pos < len(top_feats) else []
                    acts = top_acts[pos][:5] if pos < len(top_acts) else []
                    feat_str = ", ".join(
                        f"{int(f)}({a:.2f})" for f, a in zip(feats, acts, strict=True) if a > 0
                    )
                    rows_html.append(
                        f"<tr><td>{pos}{marker}</td>"
                        f"<td><code>{tok}</code></td>"
                        f"<td>{feat_str}</td></tr>"
                    )

                html = (
                    "<table><tr><th>Pos</th><th>Token</th>"
                    "<th>Top Features (id, activation)</th></tr>" + "".join(rows_html) + "</table>"
                )
                display(HTML(html))
        else:
            print("No paired SAE data found for any question.")

## 14. Trace Deep-Dive (Single Example)

Load a trace JSON to inspect the agent's reasoning steps and tool usage
for a specific run.

In [ ]:
traces_dir = RESULTS_DIR / "traces"

if traces_dir.exists() and list(traces_dir.glob("*.json")):
    example_row = df.iloc[0]
    trace_path = traces_dir / f"trace_{example_row['trace_id']}.json"

    if trace_path.exists():
        with trace_path.open(encoding="utf-8") as f:
            trace = json.load(f)

        print(f"Trace ID: {trace['trace_id']}")
        print(f"Question: {trace.get('question_text', 'N/A')}")
        print(f"Format: {trace.get('system_prompt_format', 'N/A')}")
        print(f"Total steps: {trace.get('total_steps', 0)}")
        print(f"Final answer: {trace.get('final_answer', 'N/A')[:200]}...")
        print()

        for step in trace.get("steps", []):
            print(f"--- Step {step['step_number']} ---")
            if step.get("model_response_content"):
                content_preview = step["model_response_content"][:150]
                print(f"  Model: {content_preview}...")

            for tool_exec in step.get("tool_executions", []):
                tool_name = tool_exec["tool_name"]
                args_preview = json.dumps(tool_exec["arguments"])[:100]
                result_preview = tool_exec["result"][:100]
                print(f"  Tool: {tool_name}({args_preview})")
                print(f"    -> {result_preview}")
            print()
    else:
        print(f"Trace file not found: {trace_path}")
else:
    print("No trace files found — skipping.")